# 전이학습 전략 비교 실습

이 실습에서는 **3가지 전이학습 전략**을 비교합니다:
1. **Freeze (전체 동결)**: 백본 전체를 고정하고 분류기만 학습
2. **Partial (부분 해제)**: 마지막 블록(layer4)과 분류기만 학습
3. **Full (전체 해제)**: 모든 층을 학습

**목표**: 소규모 데이터셋(3,000개)에서 각 전략의 성능을 비교하고 최적 전략을 찾기

In [ ]:
!sudo apt-get install -y fonts-nanum* | tail -n 1
!sudo fc-cache -fv
!rm -rf ~/.cache/matplotlib

In [ ]:
# 필요 라이브러리 설치

!pip install torchviz | tail -n 1
!pip install torchinfo | tail -n 1

런타임 다시 시작하세요

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib

# 나눔고딕 폰트 경로 설정
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fontprop = fm.FontProperties(fname=font_path)

# matplotlib 기본 폰트로 지정
matplotlib.rc('font', family='NanumGothic')

# 마이너스 부호 깨짐 방지
matplotlib.rcParams['axes.unicode_minus'] = False

print("한글 폰트 설정 완료:", matplotlib.rcParams['font.family'])

In [ ]:
# 필수 라이브러리 임포트
import torch  # PyTorch 메인 라이브러리
import torch.nn as nn  # 신경망 모듈
import torch.optim as optim  # 최적화 알고리즘
from torchvision import datasets, transforms, models  # 비전 관련 도구
from torch.utils.data import DataLoader, Subset  # 데이터 로더
import numpy as np  # 수치 연산
import random  # 랜덤 함수

# GPU 사용 가능 여부 확인 및 디바이스 설정
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'사용 디바이스: {device}')

# 재현성을 위한 시드 설정 (매번 동일한 결과를 얻기 위함)
torch.manual_seed(0)  # PyTorch 시드
np.random.seed(0)  # NumPy 시드
random.seed(0)  # Python random 시드

# CUDA 사용 시 추가 재현성 설정
if torch.cuda.is_available():
    torch.cuda.manual_seed(0)  # CUDA 시드
    torch.backends.cudnn.deterministic = True  # 결정적 알고리즘 사용
    torch.backends.cudnn.benchmark = False  # 벤치마크 비활성화

print('환경 설정 완료!')
print(f'PyTorch 버전: {torch.__version__}')

텍스트 전처리